# Análise de chamados_hora - Bronze Layer

Análise exploratória da tabela `bronze.chamados_hora`, feita antes do tratamento na Silver. O objetivo é identificar nulos e formatos inconsistentes nas três colunas de horário (`Hora_Abertura_Chamado`, `Hora_Inicio_Atendimento` e `Hora_Finalizacao_Atendimento`). O notebook usa apenas tabelas da Bronze.

In [0]:
from pyspark.sql import functions as F

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"

## 1. Visão geral

Consulta da tabela completa para observar o conteúdo e o formato de cada coluna.

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora

## 2. Valores nulos nas colunas de hora

Verifica quais colunas de horário têm valores nulos e quantos são.

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Abertura_Chamado is null

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento is null

In [0]:
%sql
select count(*) from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento is null

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Finalizacao_Atendimento is null

Apenas a coluna `Hora_Inicio_Atendimento` tem valores nulos. Há duas hipóteses para isso.

A primeira é que o cliente desistiu antes de ser atendido. Nesse caso, o horário de finalização deve ser igual ao de abertura, e a coluna `resolvido` da tabela `chamados` deve ser falsa.

A segunda é que o chamado não passou por atendimento humano, e o horário de início, apesar de não declarado, deve ser igual ao de abertura.

## 3. Verificação das hipóteses

A primeira query procura chamados resolvidos com horário de início nulo, cruzando com `bronze.chamados`. Como na Bronze a coluna `resolvido` ainda é texto, os valores `Sim`, `S` e `1` são considerados resolvido, o mesmo critério usado na Silver.

A segunda query procura, entre os chamados com início nulo, os que têm finalização igual à abertura. Os horários ainda estão em texto, então um mesmo horário escrito em formatos diferentes nas duas colunas não é reconhecido como igual.

In [0]:
%sql
-- essa query verifica se há casos resolvidos com horário de início nulo
-- na bronze, resolvido é texto: Sim, S e 1 significam resolvido (mesmo critério da silver)
select * from vcredit_medallion.bronze.chamados_hora a 
inner join vcredit_medallion.bronze.chamados b
on a.ID_Chamado=b.id_chamado
where a.Hora_Inicio_Atendimento is null and upper(trim(b.resolvido)) in ('SIM', 'S', '1')

In [0]:
%sql
-- essa query verifica se, nos casos de horário de início nulo, a finalização é igual à abertura
select * from vcredit_medallion.bronze.chamados_hora a 
where a.Hora_Inicio_Atendimento is null
and a.Hora_Abertura_Chamado=a.Hora_Finalizacao_Atendimento

A primeira hipótese se confirma em parte. O horário de início é nulo apenas em chamados não resolvidos, já que não existe caso de início nulo com chamado resolvido. A segunda query, porém, refuta a outra condição dessa hipótese: nos casos de início nulo, a finalização não é igual à abertura.

A segunda hipótese, de atendimento não humano com início igual à abertura, não é avaliada por essas queries.

## 4. Formato das colunas de hora

As colunas `Hora_Abertura_Chamado`, `Hora_Inicio_Atendimento` e `Hora_Finalizacao_Atendimento` precisam de formatação. A consulta abaixo mostra horários fora do padrão `dd/MM/yyyy às HH:mm:ss`, como os escritos com `-`.

In [0]:
%sql
select * from vcredit_medallion.bronze.chamados_hora where Hora_Inicio_Atendimento like "%-%"

## Conclusões e tratamentos previstos para a Silver

| Problema encontrado | Evidência | Tratamento |
|---|---|---|
| `Hora_Inicio_Atendimento` nula | Ocorre só em chamados não resolvidos (seção 3) | Silver: manter nulo. Gold: classificar como abandono |
| Finalização diferente da abertura nos casos de início nulo | Query 2 da seção 3 | Nenhum. A condição da hipótese de desistência foi descartada |
| Horários em formatos diferentes | Valores com `-` e com "às" (seção 4) | Silver: extrair data, hora e minuto por padrão de texto, independente do encoding do "à" |